# Movement branch — Stage 0 baseline

Trains the movement leg of the Shahoshi threat detector: a quantized 1D CNN that runs
in tens of KB of flash on an ESP32-S3.

This notebook is a **driver**. Every non-trivial operation lives in the `shahoshi`
package under `src/`, is unit-tested (258 tests, none needing TensorFlow), and is
reused unchanged when SisFall and the audio branch arrive. If you find yourself
writing more than a few lines in a cell here, it belongs in a module.

## What changed from `movement_model_esp32_3.ipynb`

| | before | now | why |
|---|---|---|---|
| classes | 7 (incl. `lay`) | 6 | gravity is removed, so `lay` is unidentifiable by construction: 0.25 recall float, 0.09 int8 |
| class weighting | `fit(class_weight=...)` | inside the loss | `class_weight=` reweights train loss but **not** val loss, so `val_loss` was incomparable |
| early stopping | `val_loss` | `val_macro_f1` | consequence of the above: best `val_loss` was epoch 5, so the shipped model was a 5-epoch model |
| receptive field | dilated depthwise conv | extra stride-2 stage | dilation lowers to `SPACE_TO_BATCH_ND`/`BATCH_TO_SPACE_ND`, which ESP-NN cannot accelerate |
| export | 2 `.tflite` files | 1 file, 2 outputs | the two files held a byte-identical copy of the same trunk: ~45 KB of wasted flash |
| op resolver | 8 ops hand-written | generated from the model | the model actually used 13; `AllocateTensors()` would have failed on device |
| threshold | 99th percentile | alarms per hour | at a 1.28 s hop, the 99th percentile is **28 false alarms/hour** |


## 1. Setup

On Colab: clone the repo and install it editable. Do **not** `pip install tensorflow`
here — Colab's TF is pinned against a matching NumPy, and pip cannot hot-swap a C
extension underneath a live kernel (that is the `numpy.dtype size changed` crash).
If you already did: Runtime → Restart session.

In [ ]:
REPO_URL = "https://github.com/Mursalin1011/shahoshi-model.git"

import importlib
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
_SECRET: list[str] = []          # token text to scrub from any output


def _run(cmd, cwd=None):
    """Run a command and surface its stderr on failure.

    check_call() raises with only an exit code, which is how a plain
    'repository not found' turned into an opaque CalledProcessError: exit 128.
    """
    p = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    if p.returncode != 0:
        detail = (p.stderr or p.stdout or "").strip()
        shown = " ".join(cmd)
        for s in _SECRET:                     # never echo a token back
            detail, shown = detail.replace(s, "***"), shown.replace(s, "***")
        raise RuntimeError(f"$ {shown}\nexit {p.returncode}\n{detail}")
    return p.stdout


def _auth_url():
    """Add a token from Colab Secrets, if one is configured.

    Read from the Secrets store rather than typed into a cell: a token pasted
    into a notebook is saved inside the .ipynb and travels to anyone the
    notebook is shared with.
    """
    if not IN_COLAB:
        return REPO_URL
    try:
        from google.colab import userdata

        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        return REPO_URL                        # no secret set, or access not granted
    if not token:
        return REPO_URL
    _SECRET.append(token)
    return REPO_URL.replace("https://", f"https://oauth2:{token}@")


def _find_root():
    """Nearest directory at or above cwd that is this repo. None if outside it."""
    for d in (Path.cwd(), *Path.cwd().parents):
        if (d / "src" / "shahoshi" / "__init__.py").exists():
            return d
    return None


PRIVATE_REPO_HELP = """
git could not read the repository.

Colab clones anonymously, so a PRIVATE repo fails here even though it works from
your own machine. Two ways forward:

  1. Make the repo public -- Settings -> General -> Change visibility.
     Note first that the proposal PDFs committed at the repo root carry student
     names and email addresses, which publishing would expose.

  2. Keep it private and give Colab a token:
       a. GitHub -> Settings -> Developer settings -> Personal access tokens
          -> Fine-grained tokens. Scope it to this one repo, Contents: Read-only.
       b. In Colab, click the key icon in the left sidebar, add a secret named
          GITHUB_TOKEN, paste the token there, and enable notebook access.
       c. Re-run this cell.
     Do not paste the token into a notebook cell -- it would be saved in the
     .ipynb file.
"""

# --- 1. get the code -------------------------------------------------------
root = _find_root()

if root is None:
    if not IN_COLAB:
        raise SystemExit(
            "Run this notebook from inside a checkout of the repo "
            "(expected to find src/shahoshi/ at or above the working directory)."
        )
    if not REPO_URL:
        raise SystemExit("Set REPO_URL to your GitHub repo before running on Colab.")

    name = REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
    try:
        if Path(name).exists():
            _run(["git", "fetch", "--depth", "1", "origin", "main"], cwd=name)
            _run(["git", "reset", "--hard", "origin/main"], cwd=name)
            print(f"updated existing checkout: {name}")
        else:
            _run(["git", "clone", "--depth", "1", _auth_url(), name])
            print(f"cloned: {name}")
    except RuntimeError as exc:
        text = str(exc).lower()
        if any(s in text for s in ("not found", "authentication failed",
                                   "could not read", "403", "401", "terminal prompts")):
            raise SystemExit(PRIVATE_REPO_HELP + f"\ngit said:\n{exc}") from None
        raise

    root = Path(name).resolve()

os.chdir(root)

# --- 2. make the package importable ---------------------------------------
# Two mechanisms, deliberately. `pip install -e .` is the tidy one, but on a
# long-lived Colab kernel the import finder it drops into site-packages is not
# always picked up by the already-running interpreter -- which surfaces as
# ModuleNotFoundError several cells later, far from the cause. Putting src/ on
# sys.path is what actually guarantees importability, so the install is treated
# as best-effort and the path entry is unconditional.
try:
    _run([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
    print("editable install ok")
except RuntimeError as exc:
    print(f"editable install failed (continuing via sys.path)\n{exc}\n")

src = str(root / "src")
if src not in sys.path:
    sys.path.insert(0, src)
importlib.invalidate_caches()

# --- 3. verify here, not three cells from now -----------------------------
try:
    import shahoshi
except ModuleNotFoundError as exc:
    raise SystemExit(
        f"cannot import shahoshi even with {src} on sys.path.\n"
        f"  cwd            : {Path.cwd()}\n"
        f"  src exists     : {Path(src).exists()}\n"
        f"  package exists : {(Path(src) / 'shahoshi' / '__init__.py').exists()}\n"
        f"  error          : {exc}\n"
        "If the package directory is missing, the clone is incomplete -- delete "
        "the checkout directory and re-run this cell."
    ) from None

print(f"shahoshi {shahoshi.__version__} from {Path(shahoshi.__file__).parent}")
print(f"repo root: {Path.cwd()}")
print(f"in colab : {IN_COLAB}")


In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np

from shahoshi import augment, export, manifest, quantize, scoring, splits, windows
from shahoshi.config import Config
from shahoshi.datasets import CLASSES, N_CHANNELS, N_CLASSES, load_all

CONFIG = "configs/movement.yaml"    # or configs/movement_augmented.yaml
cfg = Config.load(CONFIG)

np.random.seed(cfg.train.seed)

print(f"run          : {cfg.name}")
print(f"window       : {cfg.data.win} steps = {cfg.data.win / cfg.data.fs:.2f} s @ {cfg.data.fs} Hz")
print(f"inference hop: {cfg.data.hop_seconds:.2f} s -> {3600 / cfg.data.hop_seconds:,.0f} windows/hour")
print(f"classes      : {list(CLASSES)}")
print(f"augmentation : {'x' + str(cfg.augment.times + 1) if cfg.augment.times else 'off'}")

## 2. Load

Both corpora align on six channels — `acc_{x,y,z}` in g (gravity removed) and
`gyr_{x,y,z}` in rad/s — and both sample at 50 Hz, so no resampling.

**Mounting differs and this is the central limitation of the whole project.** UCI HAR
is a phone on the waist; MotionSense is a phone in a front trouser pocket; the device
is worn on the wrist. Axis orientation does not correspond between any of the three.
Section 4 measures the gap rather than hoping about it.

In [ ]:
ws = load_all(
    cfg.data.root,
    tags=cfg.data.datasets,
    win=cfg.data.win,
    stride=cfg.data.stride,
    download=cfg.data.download,
)

### Harmonization gate

Compare per-channel distributions across corpora **before** modelling anything. If one
corpus's spread is wildly different from another's, the merge is wrong and every number
downstream measures the mismatch instead of the activity. Silent scale disagreement is
the most common way a merged-IMU dataset goes bad.

This cell is cheap now and becomes the acceptance test for SisFall, whose data arrives
as raw ADC counts at 200 Hz with gravity still in it.

In [ ]:
from shahoshi.datasets.base import CHANNELS

tags = sorted(set(ws.dataset.tolist()))
stats = {}
for tag in tags:
    flat = ws.X[ws.dataset == tag].reshape(-1, N_CHANNELS)
    stats[tag] = {"mean": flat.mean(0), "std": flat.std(0)}

print(f"{'channel':<8s}" + "".join(f"{t + ' mean':>14s}{t + ' std':>12s}" for t in tags))
for ci, ch in enumerate(CHANNELS):
    row = f"{ch:<8s}"
    for t in tags:
        row += f"{stats[t]['mean'][ci]:>14.4f}{stats[t]['std'][ci]:>12.4f}"
    print(row)

if len(tags) > 1:
    ratios = np.array([stats[t]["std"] for t in tags])
    worst = (ratios.max(0) / np.maximum(ratios.min(0), 1e-9)).max()
    verdict = "OK" if worst < 3.0 else "SUSPECT -- investigate before training"
    print(f"\nlargest per-channel std ratio between corpora: {worst:.2f}x  [{verdict}]")

fig, axes = plt.subplots(2, 3, figsize=(14, 5), sharex=False)
for ci, ax in enumerate(axes.ravel()):
    for tag in tags:
        v = ws.X[ws.dataset == tag][:, :, ci].ravel()
        ax.hist(v, bins=120, density=True, alpha=0.5, label=tag,
                range=(np.percentile(v, 0.5), np.percentile(v, 99.5)))
    ax.set_title(CHANNELS[ci]); ax.set_yticks([])
axes[0, 0].legend(fontsize=8)
plt.suptitle("Per-channel distribution by corpus — these must overlap")
plt.tight_layout(); plt.show()

In [ ]:
# One representative window per class, plus motion energy by class. Motion energy is
# what the novelty scorer leans on hardest, so it is worth seeing directly.
fig, axes = plt.subplots(2, 3, figsize=(14, 5), sharex=True)
for ci, ax in enumerate(axes.ravel()):
    idx = np.where(ws.y_act == ci)[0]
    if not len(idx):
        ax.set_title(f"{CLASSES[ci]} (none)"); ax.axis("off"); continue
    w = ws.X[idx[len(idx) // 2]]
    t = np.arange(cfg.data.win) / cfg.data.fs
    for k, lbl in enumerate(("ax", "ay", "az")):
        ax.plot(t, w[:, k], lw=0.8, label=lbl)
    ax.set_title(CLASSES[ci]); ax.set_xlabel("s")
axes[0, 0].legend(fontsize=7)
plt.suptitle("Accelerometer, one representative window per class")
plt.tight_layout(); plt.show()

energy = np.sqrt((ws.X[:, :, :3] ** 2).sum(-1)).mean(1)
present = [i for i in range(N_CLASSES) if (ws.y_act == i).any()]
plt.figure(figsize=(9, 3.2))
try:
    plt.boxplot([energy[ws.y_act == i] for i in present],
                tick_labels=[CLASSES[i] for i in present], showfliers=False)
except TypeError:                                    # matplotlib < 3.9
    plt.boxplot([energy[ws.y_act == i] for i in present],
                labels=[CLASSES[i] for i in present], showfliers=False)
plt.ylabel("mean |acc| (g)"); plt.title("Motion energy by class")
plt.tight_layout(); plt.show()

## 3. Split by subject

Windows overlap 50%, so a random window-level split puts near-duplicate windows in both
train and test. That inflates accuracy by 5–15 points and the model falls apart on a new
wearer. `shahoshi.splits` asserts subject-disjointness and a true partition on every call;
there is no window-level option to pick by accident.

In [ ]:
if cfg.split.protocol == "subject":
    masks, groups = splits.subject_split(ws.subject, seed=cfg.split.seed,
                                         fracs=tuple(cfg.split.fracs))
else:
    masks = splits.leave_dataset_out(ws.dataset, cfg.split.test_dataset,
                                     subjects=ws.subject, seed=cfg.split.seed)

print(splits.describe(masks, ws.subject))
print("\nsplits are subject-disjoint and partition every window (asserted internally)")

tr, va, te = (ws.subset(masks[k]) for k in ("train", "val", "test"))

In [ ]:
# Normalization statistics from TRAIN ONLY. These become frozen firmware constants:
# the device cannot compute statistics over a dataset it does not have.
MEAN, STD = windows.channel_stats(tr.X)

Xtr, Xva, Xte = (windows.normalize(s.X, MEAN, STD) for s in (tr, va, te))
ytr, yva, yte = tr.y_act, va.y_act, te.y_act

print(export.normalization_source(MEAN.tolist(), STD.tolist()))

In [ ]:
# Augmentation, if the config enables it. Rotation is the only lever available against
# the waist/pocket-to-wrist mounting gap: it forces the model toward features that do
# not depend on which way the device faces. It does not close the gap -- a wrist also
# experiences genuinely different motion, not merely rotated motion -- but the
# leave-one-dataset-out delta with and without it is measurable.
if cfg.augment.times:
    aug = augment.make_augmenter(
        rotation_deg=cfg.augment.rotation_deg,
        scale_sigma=cfg.augment.scale_sigma,
        jitter_sigma=cfg.augment.jitter_sigma,
        warp_ratio=cfg.augment.warp_ratio,
        dropout_p=cfg.augment.dropout_p,
        seed=cfg.augment.seed,
    )
    Xtr_fit, (ytr_fit,) = augment.expand(Xtr, [ytr], times=cfg.augment.times, augmenter=aug)
    print(f"train set expanded {len(Xtr):,} -> {len(Xtr_fit):,} windows")
else:
    Xtr_fit, ytr_fit = Xtr, ytr
    print("augmentation off")

# Validation and test are never augmented: they must represent the real distribution.
assert len(Xva) == len(yva) and len(Xte) == len(yte)

## 4. Model

Depthwise-separable convolutions with stride-2 stages, no dilation. Global average
pooling sees the whole sequence regardless, so the dilation was buying receptive field
that GAP already provided — at the cost of two operators ESP-NN cannot accelerate.

`build` returns two views over one set of weights: a training view (supervised heads
only) and an export view that also emits the embedding for Mahalanobis scoring.

In [ ]:
import tensorflow as tf

from shahoshi.models import movement

# tf.random.set_seed alone is not enough: cuDNN picks nondeterministic
# convolution kernels, and two runs of this identical notebook gave held-out
# accuracy 0.9024 and 0.8758. A 2.7-point spread from nothing but kernel
# choice makes every ablation in this project uninterpretable.
if cfg.train.deterministic:
    quantize.set_determinism(cfg.train.seed)
else:
    tf.random.set_seed(cfg.train.seed)
print(f"TF {tf.__version__} | Keras {tf.keras.__version__} | numpy {np.__version__}")

train_model, export_model = movement.build(
    n_classes=N_CLASSES,
    win=cfg.data.win,
    channels=N_CHANNELS,
    embed_dim=cfg.model.embed_dim,
    width=cfg.model.width,
    dropout=cfg.model.dropout,
    with_fall_head=cfg.model.with_fall_head,
    bounded_relu=cfg.model.bounded_relu,
)
export_model.summary()
print(f"\ntrainable params: {train_model.count_params():,}")

## 5. Train

Class weights are applied **inside** the loss, not via `fit(class_weight=...)`. That is
the fix for the defect that made the previous run ship a 5-epoch model: `class_weight=`
reweights the training loss but not the validation loss, so `val_loss` was computed on a
different scale, diverged from epoch 5 onward while `val_accuracy` kept climbing, and
`EarlyStopping(monitor="val_loss", restore_best_weights=True)` restored epoch 5.

Weighting inside the loss makes the two comparable again; monitoring macro-F1 is a
further improvement because accuracy is dominated by the majority classes and the
classes at risk are the rare ones.

In [ ]:
cw = movement.class_weights_from(ytr_fit, N_CLASSES) if cfg.train.class_weighted else None
if cw is not None:
    print("class weights:", {CLASSES[i]: round(float(v), 3) for i, v in enumerate(cw)})

movement.compile_model(
    train_model,
    lr=cfg.train.lr,
    class_weights=cw,
    with_fall_head=cfg.model.with_fall_head,
    fall_positive_weight=cfg.train.fall_positive_weight,
)

hist = train_model.fit(
    Xtr_fit, ytr_fit,
    validation_data=(Xva, yva),
    epochs=cfg.train.epochs,
    batch_size=cfg.train.batch_size,
    verbose=2,
    callbacks=movement.default_callbacks(
        Xva, yva, n_classes=N_CLASSES,
        patience=cfg.train.patience, monitor=cfg.train.monitor,
    ),
)

In [ ]:
h = hist.history
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.5))
a1.plot(h["loss"], label="train"); a1.plot(h["val_loss"], label="val")
a1.set_title("loss (now comparable: weighting is inside the loss)")
a1.set_xlabel("epoch"); a1.legend()
a2.plot(h["val_macro_f1"], label="val macro-F1")
best = int(np.argmax(h["val_macro_f1"]))
a2.axvline(best, color="r", ls="--", label=f"restored epoch {best}")
a2.set_title("val macro-F1 (the early-stopping monitor)")
a2.set_xlabel("epoch"); a2.legend()
plt.tight_layout(); plt.show()

print(f"trained {len(h['loss'])} epochs; best val macro-F1 {max(h['val_macro_f1']):.4f} "
      f"at epoch {best} (weights restored to there)")

## 6. Float evaluation

Calibrate expectations: **high 80s on unseen subjects is a good result** for 6-class HAR
from body-frame IMU alone. Published 96%+ figures almost always come from window-level
splits or from the 561 hand-engineered features.

Confusion *within* the static group (sit/stand) is acceptable. Confusion *between* static
and dynamic is not — that is what the threat logic depends on.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score

out_f = export_model.predict(Xte, batch_size=256, verbose=0)
probs_f, emb_te = out_f[0], out_f[1]
# Keras preserves the order given to Model(inputs, outputs); assert it anyway,
# because the TFLite converter does not and the same mistake there cost a run.
assert probs_f.shape == (len(Xte), N_CLASSES), probs_f.shape
assert emb_te.shape == (len(Xte), cfg.model.embed_dim), emb_te.shape
pred_f = probs_f.argmax(1)

acc_f = float((pred_f == yte).mean())
macro_f1_f = float(f1_score(yte, pred_f, average="macro", zero_division=0))
print(f"float32 held-out accuracy : {acc_f:.4f}")
print(f"float32 held-out macro-F1 : {macro_f1_f:.4f}\n")

present = [i for i in range(N_CLASSES) if (yte == i).any() or (pred_f == i).any()]
print(classification_report(yte, pred_f, labels=present,
                            target_names=[CLASSES[i] for i in present], zero_division=0))

cm = confusion_matrix(yte, pred_f, labels=present, normalize="true")
plt.figure(figsize=(6, 5))
plt.imshow(cm, cmap="Blues", vmin=0, vmax=1)
plt.xticks(range(len(present)), [CLASSES[i] for i in present], rotation=45, ha="right")
plt.yticks(range(len(present)), [CLASSES[i] for i in present])
for r in range(len(present)):
    for c in range(len(present)):
        plt.text(c, r, f"{cm[r, c]:.2f}", ha="center", va="center", fontsize=8,
                 color="white" if cm[r, c] > 0.5 else "black")
plt.xlabel("predicted"); plt.ylabel("true")
plt.title("Confusion (row-normalized)"); plt.colorbar()
plt.tight_layout(); plt.show()

### Cross-corpus generalization

Train on one corpus, test on another. This is the honest measurement of the mounting
domain gap, and it will be substantially worse than the merged number above. It is the
figure that actually predicts wrist-worn behaviour, and it is the one the augmentation
ablation should move.

Skip on a first pass if time is short — it retrains once per held-out corpus.

In [ ]:
RUN_LEAVE_DATASET_OUT = False   # set True for the paper table

ldo_results = {}
if RUN_LEAVE_DATASET_OUT and len(cfg.data.datasets) > 1:
    for held in cfg.data.datasets:
        m = splits.leave_dataset_out(ws.dataset, held, subjects=ws.subject, seed=cfg.split.seed)
        a, b, c = (ws.subset(m[k]) for k in ("train", "val", "test"))
        mu, sd = windows.channel_stats(a.X)

        Xa = windows.normalize(a.X, mu, sd)
        ya = a.y_act
        if cfg.augment.times:
            Xa, (ya,) = augment.expand(Xa, [ya], times=cfg.augment.times, augmenter=aug)

        tm, em = movement.build(n_classes=N_CLASSES, win=cfg.data.win, channels=N_CHANNELS,
                                embed_dim=cfg.model.embed_dim, width=cfg.model.width,
                                dropout=cfg.model.dropout)
        movement.compile_model(tm, lr=cfg.train.lr,
                               class_weights=movement.class_weights_from(ya, N_CLASSES))
        tm.fit(Xa, ya, validation_data=(windows.normalize(b.X, mu, sd), b.y_act),
               epochs=cfg.train.epochs, batch_size=cfg.train.batch_size, verbose=0,
               callbacks=movement.default_callbacks(windows.normalize(b.X, mu, sd),
                                                    b.y_act, n_classes=N_CLASSES,
                                                    patience=cfg.train.patience))
        p = em.predict(windows.normalize(c.X, mu, sd), batch_size=256, verbose=0)[0].argmax(1)
        ldo_results[held] = {
            "accuracy": float((p == c.y_act).mean()),
            "macro_f1": float(f1_score(c.y_act, p, average="macro", zero_division=0)),
        }
        print(f"  held out {held:8s}: accuracy {ldo_results[held]['accuracy']:.4f}  "
              f"macro-F1 {ldo_results[held]['macro_f1']:.4f}")

    print(f"\nmerged (same-corpus) accuracy was {acc_f:.4f} -- the gap between that and "
          f"the figures above is the mounting domain gap.")
else:
    print("skipped (set RUN_LEAVE_DATASET_OUT = True)")

## 7. Quantize to int8

Weights and activations both, int8 in and int8 out, so no float kernels are linked into
the firmware. The representative dataset is stratified by class — drawn uniformly from an
imbalanced dataset, it under-represents exactly the classes whose activations get clipped.

A drop beyond about 2 points means the representative set was not diverse enough, not
that int8 is unsuitable. The previous run lost 2.8 points.

In [ ]:
rep = quantize.representative_dataset(Xtr, ytr, n=cfg.quantize.n_representative,
                                      n_classes=N_CLASSES, seed=cfg.quantize.seed)
print(f"representative set: {len(rep)} windows")

art = Path(cfg.out_dir)
blob = quantize.to_int8(export_model, rep, art / "movement_int8.tflite",
                        output_int8=cfg.quantize.output_int8)

In [ ]:
# TFLite does not preserve the Keras output order, so outputs are resolved by
# WIDTH, not by position. Reading them positionally gave int8 accuracy 0.0196
# against float 0.9024 -- argmax over the 64-wide embedding returns 0..63, which
# almost never equals a label in 0..5. The model was fine; the indexing was not.
widths = quantize.output_widths(blob)
roles = quantize.output_roles(blob, n_classes=N_CLASSES, embed_dim=cfg.model.embed_dim)
print(f"converted output widths (converter order): {widths}")
print(f"resolved roles: {roles}")
if roles["probs"] != 0:
    print("  -> the converter DID reorder the heads; positional indexing would be wrong here")

outs_q = quantize.predict_named(blob, Xte, n_classes=N_CLASSES,
                                embed_dim=cfg.model.embed_dim, batch_note=True)
probs_q, emb_q = outs_q["probs"], outs_q["embedding"]

assert probs_q.shape == (len(Xte), N_CLASSES), probs_q.shape
assert emb_q.shape == (len(Xte), cfg.model.embed_dim), emb_q.shape

delta = quantize.accuracy_delta(probs_f, probs_q, yte)
print(f"\nfloat32 : {delta['float_accuracy']:.4f}")
print(f"int8    : {delta['int8_accuracy']:.4f}   (delta {delta['delta']:+.4f})")
print(f"agreement with the float model: {delta['agreement']:.4f}")
if delta["delta"] < -0.02:
    print("\n  int8 cost more than 2 points -- widen the representative set before shipping.")

pred_q = probs_q.argmax(1)
print()
for i in present:
    msk = yte == i
    if msk.any():
        print(f"  {CLASSES[i]:<11s} recall float {(pred_f[msk] == i).mean():.3f}"
              f"  int8 {(pred_q[msk] == i).mean():.3f}  (n={int(msk.sum())})")


### Diagnosing the int8 gap

A post-training int8 conversion of a depthwise-separable network should cost
0–2 accuracy points. The first corrected run cost **8.1**, which is a real problem
rather than the price of int8.

Three causes are plausible and they need different fixes, so this measures rather
than guesses:

1. **Calibration** — the representative set misses activation ranges the test data
   produces, so activations clip. Shows up as the gap shrinking with more
   representative windows.
2. **Output resolution** — an int8 softmax has ~1/256 resolution, so the argmax flips
   whenever the top two classes sit within one quantization step. Shows up as the
   float32-output variant recovering the gap, with every kernel still integer.
3. **Weight quantization** — per-channel depthwise weight ranges too wide for int8.
   Needs quantization-aware training or an architecture change; more calibration
   data will not help.

Each conversion is fast, but this runs several, so it is behind a flag.

In [ ]:
DIAGNOSE_INT8 = True   # set False once the cause is settled

diag = None
if DIAGNOSE_INT8:
    diag = quantize.diagnose(
        export_model, Xtr, ytr, Xte, yte,
        n_classes=N_CLASSES, embed_dim=cfg.model.embed_dim,
        float_probs=probs_f, seed=cfg.quantize.seed,
    )
else:
    print('skipped')


### Are activation tails setting the quantization scale?

TFLite sizes each activation scale from the min/max seen during calibration. A layer
whose values mostly sit in [0, 2] but occasionally spike to 40 gets a scale sized for
40, and the ordinary values are squeezed into the bottom few of the 127 int8 levels.

That would explain an observation that otherwise looks backwards: enlarging the
representative set made int8 accuracy **worse** (-0.081 at 512 windows, -0.123 at
2048). More samples means more chances to observe an extreme value, a wider
calibrated range, and coarser quantization for everything else.

Read the `levels_at_p99` column: how many of the 127 levels the bulk of the
distribution actually occupies. Single digits is a problem, and
`model.bounded_relu: true` (configs/movement_relu6.yaml) is the fix.

In [ ]:
ranges = quantize.activation_ranges(export_model, Xtr, sample=1024, seed=cfg.quantize.seed)


### Weights or activations?

The previous diagnosis reached its verdict by elimination: calibration ruled out,
output dtype ruled out, therefore weights. That was not sound. Ruling those two out
leaves weight quantization *and* activation quantization both in play, and they have
very different fixes — bounded activations and a retrain for one, quantization-aware
training for the other.

The decisive variant is **dynamic range**: int8 weights, activations left in float32
at runtime, no representative dataset at all. Against full int8 it holds the weights
identical and changes only whether activations are quantized. So:

- float → dynamic_range = the cost of quantizing **weights**
- dynamic_range → full_int8 = the cost of quantizing **activations**

`weight_ranges` then checks the specific claim that depthwise per-channel ranges are
at fault, rather than asserting it. Per-channel quantization already handles spread
*between* channels; only the in-channel figure costs accuracy.

In [ ]:
ATTRIBUTE_INT8 = True   # set False once the cause is settled

attrib = None
if ATTRIBUTE_INT8:
    attrib = quantize.attribute_loss(
        export_model, Xtr, ytr, Xte, yte,
        n_classes=N_CLASSES, embed_dim=cfg.model.embed_dim,
        float_probs=probs_f,
        n_representative=cfg.quantize.n_representative,
        seed=cfg.quantize.seed,
    )
    print()
    _ = quantize.weight_ranges(export_model)
else:
    print("skipped")


In [ ]:
ops = quantize.ops_used(blob)
report = export.check_ops(ops)
arena = quantize.arena_estimate(blob)

print("operators in the converted model:")
print(" ", report["ops"])
print(f"\nunmapped        : {report['unmapped'] or 'none'}")
print(f"not ESP-NN accel: {report['not_accelerated'] or 'none'}")
print(f"\nflash {arena['flash_kb']:.1f} KB | largest tensor {arena['largest_tensor_kb']:.1f} KB")

if report["not_accelerated"]:
    print("\n  These run as reference C++ on the S3 rather than using the vector unit.")

## 8. Novelty scoring — the part that votes on threat

With no distress labels the honest formulation is one-class: learn what normal motion
looks like, score deviation. Two cheap signals, both computable on device from tensors
the model already produces — predictive entropy (free) and Mahalanobis distance from the
training embedding distribution (one 64×64 matrix; we export the Cholesky factor so the
device does a triangular solve).

### The threshold is stated in alarms per hour, not percentiles

A 99th-percentile threshold sounds like a strict 1% budget. At a 1.28 s hop that is 2812
windows per hour, so it is **28 false alarms per hour** — one every two minutes. The
wearer switches that device off, and a device that is off has zero recall whatever the
confusion matrix says.

**Every number below is a false-positive rate.** There is not one real event in this
data. Measuring detection requires labelled events, which is what Stage 1 (SisFall) is
for — do not report anything here as threat-detection performance.

In [ ]:
out_tr = export_model.predict(Xtr, batch_size=256, verbose=0)
probs_tr, emb_tr = out_tr[0], out_tr[1]
assert probs_tr.shape == (len(Xtr), N_CLASSES), probs_tr.shape
assert emb_tr.shape == (len(Xtr), cfg.model.embed_dim), emb_tr.shape

mahal = scoring.MahalanobisScorer.fit(emb_tr, shrinkage=cfg.scoring.shrinkage)

ent_tr, ent_te = scoring.entropy(probs_tr), scoring.entropy(probs_q)
mah_tr, mah_te = mahal.score(emb_tr), mahal.score(emb_te)

HOP = cfg.data.hop_seconds
print(f"inference hop {HOP:.2f} s -> {scoring.windows_per_hour(HOP):,.0f} windows/hour\n")
print(f"{'budget':>12s}  {'entropy thr':>12s} {'observed/h':>11s}   "
      f"{'mahal thr':>10s} {'observed/h':>11s}")

# The exported budget is computed unconditionally, so it does not depend on
# export_far happening to appear in the target_fars display list.
thresholds = {
    "entropy": scoring.threshold_for_far(ent_tr, cfg.scoring.export_far, HOP),
    "mahalanobis": scoring.threshold_for_far(mah_tr, cfg.scoring.export_far, HOP),
}

for far in cfg.scoring.target_fars:
    te_ = scoring.threshold_for_far(ent_tr, far, HOP)
    tm_ = scoring.threshold_for_far(mah_tr, far, HOP)
    obs_e = scoring.far_per_hour(float((ent_te > te_).mean()), HOP)
    obs_m = scoring.far_per_hour(float((mah_te > tm_).mean()), HOP)
    print(f"{far:>9.1f}/h  {te_:>12.4f} {obs_e:>11.2f}   {tm_:>10.4f} {obs_m:>11.2f}")

print(f"\n'observed/h' is measured on HELD-OUT NORMAL subjects, so it is the honest")
print(f"false-alarm rate on a new wearer. Thresholds calibrated on train data alone")
print(f"under-predict it, which is why both columns are shown.")
print(f"\nfor reference, the previous run's 99th-percentile threshold implied "
      f"{scoring.far_per_hour(0.01, HOP):.1f} alarms/hour")

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.2))
for ax, (tr_s, te_s, name, thr) in zip(
    (a1, a2),
    ((ent_tr, ent_te, "predictive entropy", thresholds["entropy"]),
     (mah_tr, mah_te, "Mahalanobis distance", thresholds["mahalanobis"])),
):
    ax.hist(tr_s, bins=60, alpha=0.6, density=True, label="train")
    ax.hist(te_s, bins=60, alpha=0.6, density=True, label="held-out")
    ax.axvline(thr, color="r", ls="--", label=f"{cfg.scoring.export_far}/h")
    ax.set_title(name); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

### Why there is no synthetic stress test here

The previous notebook perturbed held-out windows (added noise, injected impact spikes,
froze them) and reported what fraction the scorer flagged. That only confirms the scorer
reacts to *something* out of distribution. It cannot distinguish a struggle from a
motorbike over a broken road, or from dancing — and adding Gaussian noise to a
standardized window is not a physical event at all, so the numbers do not transfer.

It has been dropped rather than kept as decoration. **Stage 1 replaces it with SisFall:
real, labelled, sudden violent events**, scored with `scoring.recall_at_far` against
SisFall's own activities of daily living. That produces a genuine ROC instead of a
reaction check, and the first honest threat-detection number this project has had.

## 9. Export for ESP-IDF

The resolver is **generated from the converted model's own op list**. The previous
firmware sketch registered 8 operators by hand for a model that used 13, two of which it
had never heard of; `AllocateTensors()` would have failed on the device. Generated, it
cannot drift.

In [ ]:
art.mkdir(parents=True, exist_ok=True)

export.emit_c_array(blob, art / "movement_model_data", symbol="g_movement_model_data")

resolver_cc = export.resolver_source(ops)
(art / "movement_op_resolver.inc").write_text(resolver_cc, encoding="utf-8")
print("  wrote movement_op_resolver.inc\n")
print(resolver_cc)

export.write_config(
    art / "movement_config.json",
    win=cfg.data.win, channels=N_CHANNELS, fs=cfg.data.fs, stride=cfg.data.stride,
    classes=list(CLASSES), mean=MEAN.tolist(), std=STD.tolist(),
    thresholds=thresholds,
    extra={
        "export_far_per_hour": cfg.scoring.export_far,
        "mahalanobis_mean": mahal.mean.tolist(),
        "mahalanobis_cholesky": mahal.cholesky.tolist(),
        "operators": report["ops"],
    },
)

print()
print(export.firmware_notes(ops, arena, "movement_config.json"))

## 10. Fusion -- the vote that turns a score into an alert

A novelty score is not an alert. The proposal's claim is that **no single sensor can
raise one**: at least two of movement, heart rate and sound must agree, and keep
agreeing for four seconds, before the buzzer sounds and a message leaves the device.

That rule cannot be learned. No public corpus records movement, PPG and audio
simultaneously during real distress, so there is nothing to fit it to -- which is why it
lives in `shahoshi.fusion` as a specified state machine with tests, and is exported to
the firmware as generated C rather than transcribed by hand into a sketch.

Two things below are worth reading carefully. The **bound versus the measurement**: both
assume the branches fail independently, and they do not -- a struggle drives the movement
branch and corrupts the PPG from one physical cause. And the **reachability note**: this
build has one voter, so the vote cannot fire, and a silent field test of it measures
nothing at all.

In [ ]:
from shahoshi import fusion

rule = fusion.rule_from_config(cfg.fusion)
print(cfg.fusion.reachability_note())

# Sizing the vote needs no hardware: the arithmetic runs on each branch's
# alarms-per-hour budget, not on thresholds that do not exist yet.
planned = fusion.planned_specs(cfg.fusion)
budgets = {b.name: b.far_budget for b in cfg.fusion.branches}
fusion_bound = fusion.fused_far_upper_bound(budgets, planned, rule)
fusion_measured = fusion.measured_far(
    budgets, planned, rule, cfg.data.hop_seconds, hours=48.0)
fusion_margin = fusion.sustain_margin(planned, rule)

print()
print(f"per-branch budget : {max(budgets.values()):.1f} alarms/hour each")
print(f"fused, bound      : {fusion_bound:.2f}/h   (independence + the sustain argument)")
print(f"fused, measured   : {fusion_measured:.2f}/h   (the real engine, 48 h of Poisson fires)")
print(f"sustain margin    : {fusion_margin:+.2f} s")
if fusion_margin <= 0:
    print("  the sustain sits inside the hold, so one coincident pair of fires can alarm;")
    print("  raise fusion.sustain_seconds above the largest hold to make the clause bite")

print()
print("Both figures assume the branches fail independently. They do not: a struggle")
print("drives movement and corrupts the PPG from the same physical cause, so the field")
print("rate is higher. No public dataset lets us measure by how much.")

In [ ]:
# The movement branch votes with the Mahalanobis score, calibrated on TRAIN normal
# data at its OWN per-branch budget -- the fused budget is what consensus buys.
live = fusion.specs_from_config(
    cfg.fusion, {"movement": mah_tr}, cfg.data.hop_seconds, only_implemented=True)
for s in live:
    print(f"  {s.name:<10} threshold {s.threshold:>9.4f}   hold {s.hold_seconds:.1f}s"
          f"   weight {s.weight:g}")

print()
try:
    fusion.FusionEngine(live, rule)
    print("engine built: the configured vote is reachable with the branches that exist")
except ValueError as exc:
    print(f"engine refused, correctly: {exc}")

fusion_runtime = fusion.runtime_config(live, rule)
fusion.write_runtime_config(art / "movement_fusion.json", live, rule)

fusion_cc = fusion.fusion_source(live, rule)
(art / "shahoshi_fusion.h").write_text(fusion_cc, encoding="utf-8")
print("  wrote shahoshi_fusion.h")
print()
print(fusion_cc.split("typedef")[0])

In [ ]:
from sklearn.metrics import f1_score as _f1

metrics = {
    "float_accuracy": acc_f,
    "float_macro_f1": macro_f1_f,
    "int8_accuracy": delta["int8_accuracy"],
    "int8_macro_f1": float(_f1(yte, probs_q.argmax(1), average="macro", zero_division=0)),
    "int8_delta": delta["delta"],
    "int8_float_agreement": delta["agreement"],
    "best_val_macro_f1": float(max(hist.history["val_macro_f1"])),
    "epochs_trained": len(hist.history["loss"]),
    "restored_epoch": int(np.argmax(hist.history["val_macro_f1"])),
    "flash_kb": arena["flash_kb"],
    "n_train_windows": int(len(Xtr)),
    "n_test_windows": int(len(Xte)),
    "n_subjects": int(len(set(ws.subject.tolist()))),
    "observed_far_entropy": scoring.far_per_hour(
        float((ent_te > thresholds["entropy"]).mean()), HOP),
    "observed_far_mahalanobis": scoring.far_per_hour(
        float((mah_te > thresholds["mahalanobis"]).mean()), HOP),
    "bounded_relu": int(cfg.model.bounded_relu),
    "fusion_bound_far": fusion_bound,
    "fusion_measured_far": fusion_measured,
    "fusion_sustain_margin": fusion_margin,
}

# Fold the int8 diagnostics into the manifest rather than leaving them in cell
# output. The comparison table is what actually gets read, so the numbers that
# decide the next step belong in it.
verdict = None
if diag is not None:
    metrics["int8_delta_float_out"] = diag["float_output"]["delta"]
    if diag["sweep"]:
        best = max(diag["sweep"], key=lambda r: r["accuracy"])
        metrics["int8_delta_best_rep"] = best["delta"]
        metrics["int8_best_rep_size"] = best["rep_size"]
    verdict = diag["verdict"]

if attrib is not None:
    metrics["weight_cost"] = attrib.get("weight_cost")
    metrics["activation_cost"] = attrib.get("activation_cost")
    for _mode, _r in attrib["variants"].items():
        metrics[f"delta_{_mode}"] = _r["delta"]
    verdict = attrib["verdict"]   # direct attribution beats elimination

if ranges:
    worst = min(ranges, key=lambda r: r["levels_at_p99"])
    metrics["worst_levels_at_p99"] = worst["levels_at_p99"]
    metrics["worst_layer"] = worst["layer"]

for held, r in ldo_results.items():
    metrics[f"ldo_{held}_accuracy"] = r["accuracy"]
    metrics[f"ldo_{held}_macro_f1"] = r["macro_f1"]

manifest.write(
    cfg.report_dir, cfg.name, cfg.to_dict(), metrics,
    extra={
        "operators": report["ops"],
        "thresholds": thresholds,
        "int8_verdict": verdict,
        "attribution": {k: v for k, v in (attrib or {}).items()
                        if k != "variants"} or None,
        "activation_ranges": ranges if ranges else None,
        "fusion": fusion_runtime,
        "fusion_reachable": cfg.fusion.reachability_note(),
    },
)

print()
print(manifest.compare(cfg.report_dir, keys=[
    "float_accuracy", "int8_accuracy", "int8_delta",
    "weight_cost", "activation_cost", "worst_levels_at_p99", "bounded_relu",
]))

if verdict:
    print()
    print(f"int8 verdict: {verdict}")
if "worst_levels_at_p99" in metrics:
    print(f"worst activation layer: {metrics['worst_layer']} uses "
          f"{metrics['worst_levels_at_p99']:.1f} of 127 int8 levels")


## 11. What this baseline does and does not establish

**Established.** A 6-class activity model on unseen subjects, honestly split, quantized
to int8 with a measured cost, exported with a resolver that matches the model, and a
novelty score whose false-alarm rate is stated in alarms per hour on held-out subjects.

**Not established — and not arguable.**

1. **Nothing about threat detection.** There is not one labelled distress or violent
   event in this data. Every number above is a false-positive rate.
2. **Nothing about wrist-worn behaviour.** Every corpus is waist- or pocket-mounted. The
   leave-one-dataset-out figures in section 6 bound how badly cross-mount transfer goes;
   a wrist is worse still, because it moves differently rather than merely being rotated.
3. **Nothing about the other two branches.** The 2-of-3 consensus in the proposal needs
   heart-rate and acoustic branches. Neither exists; the vote currently has one voter,
   which section 10 states as `can_alarm = False` rather than as a quiet zero.

### Next: Stage 1, SisFall

- **Phase A** — freeze this model, score SisFall falls against SisFall ADLs with the
  entropy and Mahalanobis scorers already fitted above, and report ROC, AUPRC and
  recall at a fixed alarms-per-hour budget. No retraining. This is the cheapest
  high-value experiment available and it tells us whether the unsupervised scorer was
  ever worth anything.
- **Phase B** — add the fall head (`model.with_fall_head: true`, which the config
  refuses to enable until a fall-labelled corpus is loaded) and retrain multi-task.

The harmonization gate in section 2 is the acceptance test for SisFall: 200 Hz, raw ADC
counts, gravity still present, belt-mounted. If its channel distributions do not land on
top of UCI's, the merge is wrong and no downstream number means anything.